# TEXT SUMMARIZATION USING BART TRANSFORMER MODEL

In [ ]:
!pip install rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a3d5a232b41f53b7da46fbe49f9dee16f421f2779c76c0a0c9c49bc5634f1682
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


BART

- 1. WITHOUT FINE TUNING
- 2. WITH FINE - TUNING

#### LOADING THE DATASET

In [ ]:
import os
import torch
from rouge_score import rouge_scorer
from bert_score import score
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

folder_path = "/content/drive/MyDrive/DS310/Project/Dataset/merged_data/"

Mounted at /content/drive


In [ ]:
train_df = pd.read_csv(folder_path + "train.csv")
val_df = pd.read_csv(folder_path + "val.csv")
test_df = pd.read_csv(folder_path + "test.csv")

In [ ]:
test_df

,title,link,date_posted,headline,source,abstract,full_story,category
0,Why Some People Age Faster. And the 400 Genes ...,https://www.sciencedaily.com/releases/2025/08/...,"August 22, 2025",Why some people age faster. And the 400 genes ...,University of Colorado at Boulder,Researchers identified over 400 genes tied to ...,Scientists have uncovered more than 400 genes ...,health_medicine
1,Scientists Uncover a Multibillion-Year Epic Wr...,https://www.sciencedaily.com/releases/2024/05/...,"May 31, 2024",Scientists uncover a multibillion-year epic wr...,Tokyo Institute of Technology,Life evolved from simple geochemical processes...,The origin of life on Earth has long been a my...,fossils_ruins
2,Marfan Syndrome Increases Risk of Brain Altera...,https://www.sciencedaily.com/releases/2025/05/...,"May 16, 2025",Marfan syndrome increases risk of brain altera...,Universitat Autonoma de Barcelona,A study reveals that inflammation associated w...,A study by the Institut de Neurociències of th...,mind_brain
3,Telemedicine May Help Reduce Use of Unnecessar...,https://www.sciencedaily.com/releases/2025/02/...,"February 24, 2025",Telemedicine may help reduce use of unnecessar...,Mass General Brigham,A research team has found that telemedicine ma...,Low-value care -- medical tests and procedures...,science_society
4,"Lasers Just Unlocked a Hidden Side of Gold, Co...",https://www.sciencedaily.com/releases/2025/07/...,"July 19, 2025","Lasers just unlocked a hidden side of gold, co...",The Hebrew University of Jerusalem,Scientists have cracked a century-old physics ...,"Using only a blue laser and smart modulation, ...",matter_energy
...,...,...,...,...,...,...,...,...
1227,Simple Amino Acid Supplement Greatly Reduces A...,https://www.sciencedaily.com/releases/2025/11/...,"November 21, 2025",Simple amino acid supplement greatly reduces A...,Kindai University,Researchers discovered that the common amino a...,"New research reveals that arginine—a safe, ine...",mind_brain
1228,NaN,https://www.sciencedaily.com/releases/2025/09/...,"September 29, 2025",Black hole discovery confirms Einstein and Haw...,Simons Foundation,A fresh black hole merger detection has offere...,"When two black holes collide and merge, they r...",space_time
1229,Federal Subsidies for US Commercial Fisheries ...,https://www.sciencedaily.com/releases/2019/04/...,"April 5, 2019",Federal subsidies for US commercial fisheries ...,Duke University,A pending rule change proposed by the US Natio...,"In late 2018, the U.S. National Marine Fisheri...",business_industry
1230,Breakthrough Brain Discovery Reveals a Natural...,https://www.sciencedaily.com/releases/2025/11/...,"November 4, 2025",Breakthrough brain discovery reveals a natural...,University of Sydney,"Using powerful 7-Tesla brain imaging, research...",University of Sydney scientists have identifie...,health_medicine


In [ ]:
test_df['full_story'][0]

'Scientists have uncovered more than 400 genes linked to different forms of unhealthy aging, shedding light on why some people thrive into their 90s while others struggle decades earlier. Credit: Shutterstock\n\nIt\'s a fact of life: Some people age better than others.\nSome ease into their 90s with mind and body intact, while others battle diabetes, Alzheimer\'s or mobility issues decades earlier. Some can withstand a bad fall or bout of the flu with ease, while others never leave the hospital again.\nNew University of Colorado Boulder-led research, published this month in the journal Nature Genetics, sheds light on why that is.\nIn it, an international team of co-authors identify more than 400 genes associated with accelerated aging across seven different sub-types. The study reveals that different groups of genes underlie different kinds of disordered aging, a.k.a. frailty, ranging from cognitive decline to mobility issues to social isolation.\nThe findings lend support to what is k

In [ ]:
test_df['abstract'][0]

'Researchers identified over 400 genes tied to various forms of frailty, offering fresh insight into why people age differently. The study highlights six distinct pathways of unhealthy aging, opening the door to more precise, targeted anti-aging interventions.'

## 1. USING THE MODEL WITHOUT FINE TUNING

In [ ]:
from transformers import BartTokenizerFast, BartForConditionalGeneration

#### LOADING THE BART MODEL

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
tokenizer = BartTokenizerFast.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [ ]:
article_1 = test_df['full_story'][0]

orig_len = len(article_1.split())
max_length = int(orig_len * 0.18)
min_length = int(max_length * 0.4)
max_length = min(max_length, 150)

In [ ]:
# Clean text
def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\x00", "").replace("\ufffd", "")
    return t.strip()


# Auto-length calculation
def auto_length(text):
    orig_len = len(text.split())
    max_length = min(int(orig_len * 0.09), 150)
    min_length = int(max_length * 0.4)
    return max_length, min_length


#  Summarizer with batch
def summarize_batch_safe(texts, batch_size=2):
    summaries = []
    num_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(num_batches):
        batch_texts = texts[i*batch_size : (i+1)*batch_size]

        print(f"\n===== BATCH {i+1}/{num_batches} =====")
        print("Input size:", len(batch_texts))

        cleaned = [clean_text(t) for t in batch_texts]

        try:
            inputs = tokenizer(
                cleaned,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)
        except Exception as e:
            print("Tokenization failed:", e)
            summaries.extend([""] * len(cleaned))
            continue

        print("Max tokenized length:", inputs["input_ids"].shape[1])

        max_lens = [auto_length(t)[0] for t in cleaned]
        min_lens = [auto_length(t)[1] for t in cleaned]

        final_max_len = max(max_lens)
        final_min_len = min(min_lens)

        print("Generate max_length:", final_max_len)
        print("Generate min_length:", final_min_len)

        try:
            outputs = model.generate(
                **inputs,
                max_length=final_max_len,
                min_length=final_min_len,
                num_beams=4,
                no_repeat_ngram_size=3
            )
            batch_sum = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        except Exception as e:
            print("❌ Generation failed:", e)
            batch_sum = [""] * len(cleaned)

        for j, s in enumerate(batch_sum):
            print(f"\n--- Sample {j+1} ---")
            print(s[:300], "...")

        summaries.extend(batch_sum)

    return summaries


# Evaluation (ROUGE + BERTScore)
def evaluate_summaries(refs, preds):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    r1, r2, rl = [], [], []
    for ref, pred in zip(refs, preds):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    # BERTScore
    P, R, F = score(preds, refs, lang="en")

    return {
        "ROUGE-1": 100 * sum(r1)/len(r1),
        "ROUGE-2": 100 * sum(r2)/len(r2),
        "ROUGE-L": 100 * sum(rl)/len(rl),
        "BERTScore": 100 * float(F.mean())
    }

test_texts = test_df["full_story"].astype(str).tolist()

test_df["bart_summary"] = summarize_batch_safe(
    test_texts,
    batch_size=2
)

Streaming output truncated to the last 5000 lines.
Generate max_length: 82
Generate min_length: 21

--- Sample 1 ---
Cause and effect. We understand this concept from an early age. Tug on a pull toy's string, and the toy follows. Naturally, things get much more complicated as a system grows, as the number of variables increases, and as noise enters the picture. Eventually, it can become almost impossible to tell w ...

--- Sample 2 ---
People consistently underestimate others' desire for constructive feedback and therefore don't provide it, even when it could improve another person's performance on a task, according to research published by the American Psychological Association."People often have opportunities to provide others w ...

===== BATCH 201/616 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 52
Generate min_length: 20

--- Sample 1 ---
Water molecules are a driving force in the formation of molecular bonds, for example in proteins. Credit: INT, KIT, Constr

In [ ]:
metrics = evaluate_summaries(
    refs=test_df["abstract"].astype(str).tolist(),
    preds=test_df["bart_summary"].tolist()
)

print("\n=========== FINAL METRICS ===========")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=========== FINAL METRICS ===========
ROUGE-1: 49.9512
ROUGE-2: 36.7351
ROUGE-L: 43.0527
BERTScore: 90.2855


In [ ]:
test_df["abstract"].apply(lambda x: len(x.split())).describe()

,abstract
count,1232.000000
mean,49.444805
std,21.443449
min,9.000000
25%,33.000000
50%,47.000000
75%,64.000000
max,175.000000


In [ ]:
test_df["bart_summary"].apply(lambda x: len(x.split())).describe()

,bart_summary
count,1232.000000
mean,56.115260
std,17.367618
min,20.000000
25%,43.000000
50%,54.000000
75%,66.000000
max,124.000000


## 2. FINE - TUNING MODEL

In [ ]:
!pip install transformers datasets sentencepiece evaluate rouge_score bert_score accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    BartTokenizerFast,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import evaluate
import torch

In [ ]:
train_df = train_df[['full_story', 'abstract']]
val_df = val_df[['full_story', 'abstract']]
test_df = test_df[['full_story', 'abstract']]

train = Dataset.from_pandas(train_df)
val = Dataset.from_pandas(val_df)
test = Dataset.from_pandas(test_df)

In [ ]:
from transformers import BartTokenizerFast

tokenizer = BartTokenizerFast.from_pretrained("facebook/bart-base")

MAX_INPUT = 512
MAX_TARGET = 128

def preprocess(examples):
    # Encode input (full_story)
    model_inputs = tokenizer(
        examples["full_story"],
        max_length=MAX_INPUT,
        padding="max_length",
        truncation=True,
    )

    # Encode summary (abstract)
    labels = tokenizer(
        examples["abstract"],
        max_length=MAX_TARGET,
        padding="max_length",
        truncation=True,
    )

    # Replace padding token id with -100
    labels["input_ids"] = [
        [(lid if lid != tokenizer.pad_token_id else -100) for lid in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [ ]:
train_tokenized = train.map(
    preprocess,
    batched=True,
    remove_columns=train.column_names
)

val_tokenized = val.map(
    preprocess,
    batched=True,
    remove_columns=val.column_names
)

test_tokenized = test.map(
    preprocess,
    batched=True,
    remove_columns=test.column_names
)

Map:   0%|          | 0/9848 [00:00<?, ? examples/s]

Map:   0%|          | 0/1231 [00:00<?, ? examples/s]

Map:   0%|          | 0/1232 [00:00<?, ? examples/s]

In [ ]:
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
model.to("cuda")

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=3,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to="none",
)


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

/tmp/ipython-input-2156707883.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
trainer.train()

Step,Training Loss
50,0.799300
100,1.619100
150,1.556400
200,1.484800
250,1.595200
300,1.519400
350,1.412100
400,1.480300
450,1.314200
500,1.416100


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=7386, training_loss=1.042325714755891, metrics={'train_runtime': 1930.0801, 'train_samples_per_second': 15.307, 'train_steps_per_second': 3.827, 'total_flos': 9007026961121280.0, 'train_loss': 1.042325714755891, 'epoch': 3.0})

In [ ]:
def auto_length(text):
    orig_len = len(text.split())
    max_length = min(int(orig_len * 0.09), 150)
    min_length = max(5, int(max_length * 0.4))
    return max_length, min_length

def summarize(text):
    max_l, min_l = auto_length(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    output = model.generate(
        **inputs,
        max_length=max_l,
        min_length=min_l,
        num_beams=4,
        no_repeat_ngram_size=3,
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
model_path = "/content/drive/MyDrive/DS310/Project/checkpoint-7000-BART"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)

In [ ]:
def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\x00", "").replace("\ufffd", "")
    return t.strip()

In [ ]:
def auto_length(text):
    orig_len = len(text.split())

    max_length = min(int(orig_len * 0.09), 150)
    max_length = max(max_length, 30)

    min_length = int(max_length * 0.35)
    min_length = min(min_length, max_length - 5)

    return max_length, min_length

In [ ]:
def summarize_batch_safe(texts, batch_size=2):
    summaries = []
    num_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(num_batches):
        batch_texts = texts[i*batch_size:(i+1)*batch_size]
        print(f"\n===== BATCH {i+1}/{num_batches} =====")
        print("Input size:", len(batch_texts))

        cleaned = [clean_text(t) for t in batch_texts]

        # Tokenize
        try:
            inputs = tokenizer(
                cleaned,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            ).to(device)
        except Exception as e:
            print("Tokenization failed:", e)
            summaries.extend([""] * len(cleaned))
            continue

        print("Max tokenized length:", inputs["input_ids"].shape[1])

        # Auto lengths
        max_lens = [auto_length(t)[0] for t in cleaned]
        min_lens = [auto_length(t)[1] for t in cleaned]

        final_max_len = min(max(max_lens), 150)
        final_min_len = max(min_lens)

        if final_min_len >= final_max_len:
            final_min_len = max(5, final_max_len - 10)

        print("Generate max_length:", final_max_len)
        print("Generate min_length:", final_min_len)

        # Generate
        try:
            outputs = model.generate(
                **inputs,
                max_length=final_max_len,
                min_length=final_min_len,
                num_beams=4,
                no_repeat_ngram_size=3
            )
            batch_sum = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        except Exception as e:
            print("Generation failed:", e)
            batch_sum = [""] * len(cleaned)

        # Print
        for j, s in enumerate(batch_sum):
            print(f"\n--- Sample {j+1} ---")
            print(s[:300], "...")

        summaries.extend(batch_sum)

    return summaries


In [ ]:
def evaluate_summaries(refs, preds):
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    r1 = []
    r2 = []
    rl = []

    for ref, pred in zip(refs, preds):
        scores = scorer.score(ref, pred)
        r1.append(scores["rouge1"].fmeasure)
        r2.append(scores["rouge2"].fmeasure)
        rl.append(scores["rougeL"].fmeasure)

    P, R, F = score(preds, refs, lang="en")

    return {
        "ROUGE-1": 100 * sum(r1) / len(r1),
        "ROUGE-2": 100 * sum(r2) / len(r2),
        "ROUGE-L": 100 * sum(rl) / len(rl),
        "BERTScore": 100 * float(F.mean())
    }

In [ ]:
test_texts = test_df["full_story"].astype(str).tolist()

test_df["fine_tune_bart_summary"] = summarize_batch_safe(
    test_texts,
    batch_size=2
)

Streaming output truncated to the last 5000 lines.
Generate max_length: 82
Generate min_length: 28

--- Sample 1 ---
A new mathematical tool can tease out the contributions that each variable in a system makes to a measured effect -- both separately and, more importantly, in combination. ...

--- Sample 2 ---
People consistently underestimate others' desire for constructive feedback and therefore don't provide it, even when it could improve another person's performance on a task, according to new research. ...

===== BATCH 201/616 =====
Input size: 2
Max tokenized length: 512
Generate max_length: 52
Generate min_length: 18

--- Sample 1 ---
Water doesn’t just circulate freely—it actively binds molecules inside tiny molecular cavities. Scientists discovered that trapped water acts like a passive bystander, allowing molecules to bind more strongly. This “highly energetic” state explains why molecules ...

--- Sample 2 ---
Psychologists found that the brains of children from areas contain

In [ ]:
metrics = evaluate_summaries(
    refs=test_df["abstract"].astype(str).tolist(),
    preds=test_df["fine_tune_bart_summary"].tolist()
)

print("\n=========== FINAL METRICS ===========")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=========== FINAL METRICS ===========
ROUGE-1: 60.2872
ROUGE-2: 47.7084
ROUGE-L: 53.9225
BERTScore: 92.9393


# Factual consistency

In [ ]:
from transformers import pipeline
import torch

nli = pipeline(
    "text-classification",
    model="roberta-large-mnli",
    return_all_scores=False,
    device=0 if torch.cuda.is_available() else -1
)

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [ ]:
def factual_consistency_score(inputs, summaries, max_chars=600):
    labels = []
    scores = []

    for inp, summ in zip(inputs, summaries):
        if not summ or not inp:
            labels.append("EMPTY")
            scores.append(0.0)
            continue

        premise = inp[:max_chars]
        hypothesis = summ

        try:
            result = nli({
                "text": premise,
                "text_pair": hypothesis
            })

            if isinstance(result, list):
                result = result[0]

            labels.append(result["label"])
            scores.append(result["score"])

        except Exception as e:
            print("NLI ERROR:", repr(e))
            labels.append("ERROR")
            scores.append(0.0)

    return labels, scores

In [ ]:
print(nli({
    "text": test_df["full_story"].iloc[0][:600],
    "text_pair": test_df["bart_summary"].iloc[0]
}))

{'label': 'NEUTRAL', 'score': 0.491384357213974}


In [ ]:
labels_base, scores_base = factual_consistency_score(
    test_df["full_story"].tolist(),
    test_df["bart_summary"].tolist()
)

labels_ft, scores_ft = factual_consistency_score(
    test_df["full_story"].tolist(),
    test_df["fine_tune_bart_summary"].tolist()
)

test_df["nli_base"] = labels_base
test_df["nli_ft"] = labels_ft

In [ ]:
print("BART-base NLI:")
print(test_df["nli_base"].value_counts(normalize=True) * 100)

print("\nFine-tuned BART NLI:")
print(test_df["nli_ft"].value_counts(normalize=True) * 100)

BART-base NLI:
nli_base
NEUTRAL          60.714286
ENTAILMENT       37.824675
CONTRADICTION     1.461039
Name: proportion, dtype: float64

Fine-tuned BART NLI:
nli_ft
NEUTRAL          73.376623
ENTAILMENT       25.324675
CONTRADICTION     1.298701
Name: proportion, dtype: float64


# Compression & Coverage

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download("punkt")

from nltk.util import ngrams

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
def compression_ratio(input_text, summary):
    if not input_text or not summary:
        return 0.0
    return len(summary.split()) / max(len(input_text.split()), 1)

In [ ]:
def coverage(input_text, summary):
    if not input_text or not summary:
        return 0.0

    input_words = set(input_text.lower().split())
    summary_words = summary.lower().split()

    return sum(w in input_words for w in summary_words) / len(summary_words)

In [ ]:
def redundancy(summary, n=3):
    if not summary:
        return 0.0

    tokens = summary.split()
    if len(tokens) < n:
        return 0.0

    ng = list(ngrams(tokens, n))
    return 1 - len(set(ng)) / len(ng)

In [ ]:
def analyze_style(df, col):
    return {
        "compression": df.apply(
            lambda x: compression_ratio(x["full_story"], x[col]), axis=1
        ).mean(),
        "coverage": df.apply(
            lambda x: coverage(x["full_story"], x[col]), axis=1
        ).mean(),
        "redundancy": df[col].apply(redundancy).mean()
    }

style_base = analyze_style(test_df, "bart_summary")
style_ft = analyze_style(test_df, "fine_tune_bart_summary")

print("BART-base:", style_base)
print("Fine-tuned:", style_ft)
def analyze_style(df, col):
    return {
        "compression": df.apply(
            lambda x: compression_ratio(x["full_story"], x[col]), axis=1
        ).mean(),
        "coverage": df.apply(
            lambda x: coverage(x["full_story"], x[col]), axis=1
        ).mean(),
        "redundancy": df[col].apply(redundancy).mean()
    }

style_base = analyze_style(test_df, "bart_summary")
style_ft = analyze_style(test_df, "fine_tune_bart_summary")

print("BART-base:", style_base)
print("Fine-tuned:", style_ft)

BART-base: {'compression': np.float64(0.08734836606809604), 'coverage': np.float64(0.9712237671991447), 'redundancy': np.float64(0.0002247580708683902)}
Fine-tuned: {'compression': np.float64(0.06795510018286195), 'coverage': np.float64(0.9497975030839589), 'redundancy': np.float64(0.0)}
BART-base: {'compression': np.float64(0.08734836606809604), 'coverage': np.float64(0.9712237671991447), 'redundancy': np.float64(0.0002247580708683902)}
Fine-tuned: {'compression': np.float64(0.06795510018286195), 'coverage': np.float64(0.9497975030839589), 'redundancy': np.float64(0.0)}


Nhận xét:
- Kết quả phân tích cho thấy mô hình BART sau khi fine-tune tạo ra các bản tóm tắt ngắn gọn hơn và ít lặp ý hơn so với mô hình BART-base ban đầu. Mặc dù độ bao phủ nội dung của mô hình fine-tuned giảm nhẹ, giá trị này vẫn ở mức cao, cho thấy mô hình không bịa thông tin mà có xu hướng trừu tượng hóa nội dung tốt hơn. Tuy nhiên, tỷ lệ nén thấp cũng cho thấy nguy cơ mô hình bỏ sót một số chi tiết quan trọng trong các văn bản dài.

# Test Model

In [ ]:
def show_abstract_comparisons(df, ref_col, pred_col, n=20):
    for i in range(min(n, len(df))):
        print(f"\n SAMPLE {i+1}")

        print(" ORIGINAL ABSTRACT:")
        print(df.loc[i, ref_col])

        print("\n GENERATED ABSTRACT (Fine-tuned BART):")
        print(df.loc[i, pred_col])


In [ ]:
show_abstract_comparisons(
    df=test_df,
    ref_col="abstract",
    pred_col="fine_tune_bart_summary",
    n=20
)


 SAMPLE 1
 ORIGINAL ABSTRACT:
Researchers identified over 400 genes tied to various forms of frailty, offering fresh insight into why people age differently. The study highlights six distinct pathways of unhealthy aging, opening the door to more precise, targeted anti-aging interventions.

 GENERATED ABSTRACT (Fine-tuned BART):
Scientists have uncovered 400 genes linked to different forms of unhealthy aging, shedding light on why some people thrive into their 90s while others struggle decades earlier. The study reveals that different groups of genes underlie different kinds of disordered aging, a.k.a. frailty, ranging from cognitive decline to mobility issues to social

 SAMPLE 2
 ORIGINAL ABSTRACT:
Life evolved from simple geochemical processes present on the early Earth. However, the details of this transformation remain a mystery. In this study, researchers modeled the emergence of metabolic pathways from simple geochemical precursors and concluded that surprisingly few reactions i

In [ ]:
!cp -r ./bart_finetuned/checkpoint-7000 /content/drive/MyDrive/